In [ ]:
# %load init_ipynb
import os
import matplotlib.pyplot as pl

import sys
assert os.path.basename(sys.prefix) == 'wakai'

plotpar = {'axes.labelsize': 16,
           'font.size': 16,
           'legend.fontsize': 16,
           'xtick.labelsize': 16,
           'ytick.labelsize': 16,
           'text.usetex': False}
pl.rcParams.update(plotpar)

In [ ]:
#!{sys.executable} -m pip install --upgrade latex

In [ ]:
import sys

try: 
    import isochrones
except:
    !{sys.executable} -m pip install --upgrade isochrones
    import isochrones

assert isochrones.__version__ == "2.1"

In [ ]:
# See page 5 in http://waps.cfa.harvard.edu/MIST/README_tables.pdf

eep_dict = {
    1: 'pre-main sequence (PMS)',
    202: 'zero age main sequence (ZAMS)',
    353: 'intermediate age main sequence (IAMS )',
    454: 'terminal age main sequence (TAMS)',
    605: 'tip of the red giant branch (RGBTip)',
    631: 'zero age core helium burning (ZACHeB)',
    707: 'terminal age core helium burning (TACHeB)'
}

In [ ]:
from isochrones.mist import MISTIsochroneGrid


iso_grid = MISTIsochroneGrid()
iso_grid.df.head()

In [ ]:
#get unique indices

iso_grid_indices = {}
for i in range(3):
    index = iso_grid.df.index.get_level_values(i)
    iso_grid_indices[index.name] = index.unique()

In [ ]:
import matplotlib.pyplot as pl

feh0 = 0.0
logAges = [7.0,8.0,9.0]

fig, ax = pl.subplots(dpi=100)

x, y = 'logTeff', 'logL'
for age in logAges:
    iso_df = iso_grid.df.xs((age, feh0), level=(0,1))
    #plot isochrones
    iso_df.plot(x, y, label=f"{age:.1f}", ax=ax)
    #mark EEPs
    eep_indices = iso_df.index.isin(eep_dict.keys())
    #plot EEPs
    iso_df[eep_indices].plot.scatter(x, y, c='r', ax=ax)
    ax.legend(title=f"log(age)")
    ax.invert_xaxis()

In [ ]:
import sys

try: 
    import astroquery
except:
    !{sys.executable} -m pip install --upgrade astroquery
    import astroquery

assert astroquery.__version__ == "0.4.6"

In [ ]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
import pandas as pd

clobber = False

table_name = "ps" #self-consistent set of parameters
# table_name = "pscomppars" #a more complete, though not necessarily self-consistent set of parameters
fp = f"../data/nexsci_{table_name}.csv"
if clobber:
    nexsci_tab = NasaExoplanetArchive.query_criteria(table=table_name, where="discoverymethod like 'Transit'")
    df_nexsci = nexsci_tab.to_pandas()
    df_nexsci.to_csv(fp, index=False)
    print("Saved: ", fp)
else:
    df_nexsci = pd.read_csv(fp)
    print("Loaded: ", fp)

In [ ]:
df_nexsci.shape

In [ ]:
df_nexsci.query("hostname=='V1298 Tau'")

In [ ]:
eep_dict

In [ ]:
df_nexsci.disc_facility.unique()

In [ ]:
import sys  
sys.path.insert(0, '..')

In [ ]:
from utils import get_relative_err_index

idx = (df_nexsci.st_age<1) & (df_nexsci.default_flag==1) #& (df_nexsci.disc_facility=='Kepler') 
idx2 = get_relative_err_index(df_nexsci[idx], par='st_age', rel_err=0.2)

df_young = df_nexsci[idx][idx2]
young_hoststar_names = df_young.hostname.unique()
len(young_hoststar_names), young_hoststar_names

In [ ]:
import numpy as np

df_young['logTeff'] = df_young.st_teff.apply(lambda x: np.log10(x))
df_young['logL'] = df_young['st_lum'].copy()

In [ ]:
import numpy as np
import matplotlib.pyplot as pl

def plot_iso_grid(x='logTeff', y='logL', 
                eep1=202, eep2=353, 
                logAges=[7.0,8.0,9.0],
                feh0 = 0.0,
                ax=None):
    if ax is None:
        fig, ax = pl.subplots(dpi=150)

    stage_start = eep_dict[eep1]
    stage_end = eep_dict[eep2]

    for age in logAges:
        iso_df = iso_grid.df.xs((age, feh0), level=(0,1))
        iso_df_eeps = np.array(iso_df.index.tolist())
        idx = (iso_df_eeps>eep1) & (iso_df_eeps<=eep2)
        iso_df[idx].plot(x, y, label=f"{age:.1f}", ax=ax)
        ax.set_ylabel(y)
        ax.set_title(f"from {stage_start}\nto {stage_end}")
        ax.legend(title="log(age)")
        if x=='logTeff':
            ax.invert_xaxis()
    return ax

def plot_planets(df,
                x='logTeff', y='logL', 
                ax=None):
    if ax is None:
        fig, ax = pl.subplots(dpi=150)
    df.plot.scatter(x=x, y=y, ax=ax)
    return ax
    
ax = plot_iso_grid()

_ = plot_planets(df_young, ax=ax)

In [ ]:
df_young['mass'] = df_young['st_mass'].copy()
df_young['radius'] = df_young['st_rad'].copy()

x, y = 'mass', 'radius'

ax = plot_iso_grid(x=x, y=y)

_ = plot_planets(df_young, x=x, y=y, ax=ax)
ax.set_xlim(0, 3)
ax.set_ylim(0, 3)

In [ ]:
df_young['mass'] = df_young['st_mass'].copy()
df_young['radius'] = df_young['st_rad'].copy()

x, y = 'mass', 'radius'

ax = plot_iso_grid(x=x, y=y)

_ = plot_planets(df_young, x=x, y=y, ax=ax)
ax.set_xlim(0, 3)
ax.set_ylim(0, 3)